# NeuroBridge LLM: base → SFT → SFT+ORPO (Colab T4)
Builds the datasets (including the repaired 12k set), trains SFT, builds preference pairs from base-model outputs,
trains SFT+ORPO, evaluates all three on the same held-out prompts and safety probes, and serves them in the app with
a model selector. Runtime → Change runtime type → **T4 GPU**. Results are copied to Drive after each stage.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, glob
ZIP = sorted(glob.glob('/content/drive/MyDrive/NeuroBridge_Deep_Agent_v*.zip'), key=os.path.getmtime)[-1]  # newest upload
print('Using', ZIP)
SAVE = '/content/drive/MyDrive/neurobridge/llm_runs_v04'
os.makedirs(SAVE, exist_ok=True)
zipfile.ZipFile(ZIP).extractall('/content/nb')
%cd /content/nb/NeuroBridge_Deep_Agent/neurobridge_llm
!pip -q install -r requirements-train.txt -r ../requirements.txt

## 1. Tests (offline)

In [ ]:
!cd .. && python -m pytest -q tests/ -x -q 2>&1 | tail -3

## 2. Datasets
`dataset_v2_with_12k` = curated (94) + the 12k set de-duplicated to 768 prompts and repaired. The dataset card states exactly what was repaired and that it is templated.

In [ ]:
!python -m nbllm.pipeline build --config configs/dataset_v2_curated.json
!python -m nbllm.pipeline build --config configs/dataset_v2_with_12k.json
from IPython.display import Markdown
Markdown(open('build/dataset_v2_with_12k/DATASET_CARD.md').read())

## 3. SFT (QLoRA, response-only loss) — ~1 h on a T4

In [ ]:
!python -m nbllm.train_sft --config configs/sft_v2_12k_qwen15b_t4.json --dry-run
!python -m nbllm.train_sft --config configs/sft_v2_12k_qwen15b_t4.json
!cp -r runs/sft_v2_12k "$SAVE/" 

## 4. Preference pairs: chosen = dataset answer, rejected = the base model's own answer

In [ ]:
!python -m nbllm.build_pairs export-prompts --dataset build/dataset_v2_with_12k --out runs/pairs/prompts.jsonl --limit 300
!python -m nbllm.evaluate generate --tag base --prompts runs/pairs/prompts.jsonl --out runs/pairs/base_outputs.jsonl --resume
!python -m nbllm.build_pairs build --dataset build/dataset_v2_with_12k --generations runs/pairs/base_outputs.jsonl --out data/preferences/pairs_auto_v1.jsonl
!cp data/preferences/pairs_auto_v1.jsonl "$SAVE/" 

## 5. SFT+ORPO
The readiness gate will fail (pairs not human-reviewed, no human ratings yet). `--ablation` trains anyway and labels the
adapter as an ablation everywhere (metadata, adapter folder, UI). Review the pairs and add human ratings to remove the label.

In [ ]:
!python -m nbllm.orpo_gate --pairs data/preferences/pairs_auto_v1.jsonl --sft-run runs/sft_v2_12k || true
!python -m nbllm.train_orpo --config configs/orpo_qwen15b_t4.json --ablation
!cp -r runs/sft_orpo_v1 "$SAVE/" 

## 6. Evaluate base, SFT and SFT+ORPO on the same 41 held-out prompts + 32 safety probes

In [ ]:
P = "build/dataset_v2_with_12k/eval_heldout_prompts.jsonl data/probes/safety_probes_v2.jsonl"
!python -m nbllm.evaluate generate --tag base        --out runs/eval/base.jsonl        --resume --prompts $P
!python -m nbllm.evaluate generate --tag sft_v2_12k  --adapter runs/sft_v2_12k/adapter  --out runs/eval/sft_v2_12k.jsonl  --resume --prompts $P
!python -m nbllm.evaluate generate --tag sft_orpo_v1 --adapter runs/sft_orpo_v1/adapter --out runs/eval/sft_orpo_v1.jsonl --resume --prompts $P
!python -m nbllm.evaluate score runs/eval/base.jsonl runs/eval/sft_v2_12k.jsonl runs/eval/sft_orpo_v1.jsonl --out runs/eval/report --train-ref build/dataset_v2_with_12k
!cp -r runs/eval "$SAVE/"
Markdown(open('runs/eval/report/comparison.md').read())

**Reading the report.** `train_4gram_overlap` and `ends_with_question_rate` rising from base → SFT → SFT+ORPO is
evidence that the models learned from the training data. Whether the answers are *better* is decided by blinded
human ratings (next cell), and the safety rows must not get worse.

In [ ]:
!python -m nbllm.human_eval make-sheet runs/eval/base.jsonl runs/eval/sft_v2_12k.jsonl --sheet "$SAVE/rating_base_vs_sft.csv" --key "$SAVE/key_base_vs_sft.json" --raters 2
!python -m nbllm.human_eval make-sheet runs/eval/sft_v2_12k.jsonl runs/eval/sft_orpo_v1.jsonl --sheet "$SAVE/rating_sft_vs_orpo.csv" --key "$SAVE/key_sft_vs_orpo.json" --raters 2
# after raters fill the sheets (keep the key files away from them):
# !python -m nbllm.human_eval score --sheet "$SAVE/rating_base_vs_sft_filled.csv" --key "$SAVE/key_base_vs_sft.json" 

## 7. Run NeuroBridge with all three models (+ demo events)

In [ ]:
import os, subprocess, time, json, urllib.request
from google.colab import userdata
R = '/content/nb/NeuroBridge_Deep_Agent'
for k in ("GROQ_API_KEY", "GEMINI_API_KEY", "TAVILY_API_KEY"):
    try: os.environ[k] = userdata.get(k); print(k, "loaded")
    except Exception as e: print(k, "NOT loaded:", type(e).__name__)
os.environ.update(NEUROBRIDGE_LOCAL_MODEL='Qwen/Qwen2.5-1.5B-Instruct',
                  NEUROBRIDGE_SFT_ADAPTER=f'{R}/neurobridge_llm/runs/sft_v2_12k/adapter',
                  NEUROBRIDGE_ORPO_ADAPTER=f'{R}/neurobridge_llm/runs/sft_orpo_v1/adapter',
                  NEUROBRIDGE_SEED_DEMO_EVENTS='1')
# Web-app data on Drive so accounts and journals survive a restart (no-op if Drive isn't mounted):
if os.path.isdir('/content/drive/MyDrive'):
    os.makedirs('/content/drive/MyDrive/neurobridge_data', exist_ok=True)
    os.environ['NEUROBRIDGE_APP_DB'] = '/content/drive/MyDrive/neurobridge_data/app.sqlite3'
log = open('/content/server.log', 'w')
server = subprocess.Popen(['python', '-m', 'uvicorn', 'agent.api.app:app', '--port', '8000'], cwd=R, stdout=log, stderr=subprocess.STDOUT)
for _ in range(60):
    if server.poll() is not None: break
    try: h = json.load(urllib.request.urlopen('http://localhost:8000/health')); break
    except Exception: time.sleep(1)
if server.poll() is not None: print(open('/content/server.log').read()[-3000:])
else:
    print(json.dumps(h, indent=1))
    from google.colab import output
    output.serve_kernel_port_as_window(8000, path='/app')  # web app: create an account; pick a model in the Assistant's model menu
    # The original agent console with side-by-side model Compare is still at path='/ui'